# Study 1: example images for the concept audit (post hoc, illustrative)

For three NIH test images this notebook lists the concepts that contributed most to CLEAR's concept-route score, using the same formula and relevance patterns as Amendment 1. The images were drawn before this notebook was written: one true positive each for cardiomegaly, emphysema and hernia, picked at random (seed 2026) from all true positives at the held-out threshold. The rule and the picks are recorded in the project notes.

The notebook first recomputes each image's score and stops if it does not match the score saved by the registered run.

An optional last part checks whether any VinDr-CXR DICOM header records ethnicity. It runs only if the VinBigData competition is attached.

**Settings:** Accelerator **GPU T4** (one GPU is enough), Internet **On**.
**Inputs:** the dataset **NIH Chest X-rays** (as in the main run). Optional: the competition **VinBigData Chest X-ray Abnormalities Detection**.
**Run:** Save Version, then Save & Run All (Commit). About 15 to 20 minutes. Then download `fig7_outputs.zip` from the Output tab and upload it here.


In [ ]:
# 0. Settings, packages and the fixed selection
import os, sys, time, json, glob, re, subprocess, platform
os.environ["HF_HOME"] = "/tmp/hf"; os.environ["TOKENIZERS_PARALLELISM"] = "false"
SEED = 2026; T0 = time.time()
OUT = "/kaggle/working/fig7_out"; os.makedirs(OUT, exist_ok=True)
CLEAR_COMMIT = "57d2eae57ddabf8e56c8013ec675336d71b3d8e7"
HF_SHA = "142f169c8b006b0f28cf55c6a237c3003db5ed8f"
SELECTION = {
 "Cardiomegaly": {
  "image": "00022215_010.png",
  "n_tp": 1007,
  "n_pos": 1069,
  "thr": 0.496964,
  "score": 0.500447,
  "labels": "Cardiomegaly|Edema",
  "age": 53,
  "sex": "F",
  "view": "AP"
 },
 "Emphysema": {
  "image": "00009349_001.png",
  "n_tp": 949,
  "n_pos": 1093,
  "thr": 0.495967,
  "score": 0.500476,
  "labels": "Edema|Effusion|Emphysema|Infiltration|Mass",
  "age": 70,
  "sex": "M",
  "view": "PA"
 },
 "Hernia": {
  "image": "00000003_005.png",
  "n_tp": 71,
  "n_pos": 86,
  "thr": 0.489787,
  "score": 0.490458,
  "labels": "Hernia",
  "age": 78,
  "sex": "F",
  "view": "PA"
 }
}
FIND = list(SELECTION)                                  # Cardiomegaly, Emphysema, Hernia
WORD = {"Cardiomegaly": "cardiomegaly", "Emphysema": "emphysema", "Hernia": "hernia"}
POS = [WORD[f] for f in FIND]; NEG = ["no " + w for w in POS]   # registered template T1
RELEVANCE = {"Cardiomegaly": r"cardiomegal|heart size|enlarged heart|cardiac silhouette|cardiac enlargement|heart is enlarged|cardiac size",
             "Emphysema": r"emphysem", "Hernia": r"hernia"}  # copied from Amendment 1
!rm -rf /kaggle/working/CLEAR && git clone -q https://github.com/peterhan91/CLEAR.git /kaggle/working/CLEAR
!git -C /kaggle/working/CLEAR checkout -q 57d2eae57ddabf8e56c8013ec675336d71b3d8e7
!pip install -q /kaggle/working/CLEAR
import numpy as np, pandas as pd, torch, torch.nn.functional as F, transformers
from PIL import Image
run_info = {"started_utc": time.strftime("%Y-%m-%d %H:%M:%S", time.gmtime()), "seed": SEED,
            "clear_git_commit": subprocess.run(["git", "-C", "/kaggle/working/CLEAR", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip(),
            "hf_revision": HF_SHA, "python": platform.python_version(), "torch": torch.__version__,
            "transformers": transformers.__version__, "gpus": [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],
            "selection": SELECTION, "prompts_pos": POS, "prompts_neg": NEG}
assert run_info["clear_git_commit"] == CLEAR_COMMIT, run_info["clear_git_commit"]
assert torch.cuda.is_available(), "No GPU: set Accelerator to GPU T4"
print(json.dumps({k: v for k, v in run_info.items() if k != "selection"}, indent=1))


In [ ]:
# 1. Find the three images in the NIH dataset
def find_file(name):
    for root, dirs, files in os.walk("/kaggle/input"):
        dirs[:] = [d for d in dirs if not d.startswith("images_")]
        if name in files: return os.path.join(root, name)
    raise FileNotFoundError(name + ": attach the NIH Chest X-rays dataset")
base = os.path.dirname(find_file("Data_Entry_2017.csv"))
PATH = {}
for f in FIND:
    hits = glob.glob(os.path.join(base, "images_*", "images", SELECTION[f]["image"]))
    assert len(hits) == 1, (f, hits); PATH[f] = hits[0]
test_ids = set(open(os.path.join(base, "test_list.txt")).read().split())
entry = pd.read_csv(os.path.join(base, "Data_Entry_2017.csv")).set_index("Image Index")
for f in FIND:
    im = SELECTION[f]["image"]
    assert im in test_ids, im                                   # official test split
    assert f in entry.loc[im, "Finding Labels"].split("|"), (im, entry.loc[im, "Finding Labels"])
    print(f, im, entry.loc[im, "Finding Labels"], Image.open(PATH[f]).size)


In [ ]:
# 2. Concept bank (pinned Hugging Face revision) and the registered SFR-Embedding-Mistral prompt embeddings
#    The prompt embeddings are the ones saved by the registered NIH run (plain text, template T1), embedded here as bytes.
import base64, hashlib
from huggingface_hub import hf_hub_download
from clear.concepts import ConceptBank
dl = lambda n: hf_hub_download("peterhan91/CLEAR", n, revision=HF_SHA)
bank = ConceptBank.from_files(dl("mimic_concepts.csv"), clip_embeddings_path=dl("concept_embeddings_368294.pt"),
                              llm_embeddings_path=dl("concepts_embeddings_sfr_mistral.pickle"), allow_unsafe_pickle=True)
C_llm = bank.ensure_llm_embeddings(); C_clip = torch.from_numpy(bank.ensure_clip_embeddings()).float()
run_info["n_concepts"] = len(bank.concepts); print("concepts:", run_info["n_concepts"], C_clip.shape, C_llm.shape)
EMB_B64 = """
...
"""
raw = base64.b64decode("".join(EMB_B64.split()))
assert hashlib.sha256(raw).hexdigest() == "7801c04d698753605f4da2b39f63af6f25495b8e0bc781fe62201fdcff2dac6a"
EMB = np.frombuffer(raw, dtype=np.float32).reshape(2, len(FIND), 4096)
P_sfr, N_sfr = EMB[0].copy(), EMB[1].copy()
run_info["prompt_embedding_sha256"] = hashlib.sha256(raw).hexdigest()
print("registered prompt embeddings:", P_sfr.shape, N_sfr.shape)


In [ ]:
# 3. CLEAR (pinned revision), concept projection M, and the registered score check
from clear import load_pretrained
dev = "cuda:0"
clear_model, clear_pre = load_pretrained(device=dev, revision=HF_SHA); clear_model.eval()
M = torch.zeros(C_clip.shape[1], C_llm.shape[1], dtype=torch.float64, device=dev)
C_llm_t = torch.from_numpy(C_llm)
for i in range(0, C_clip.shape[0], 8192):
    M += F.normalize(C_clip[i:i+8192].to(dev), dim=1).double().T @ C_llm_t[i:i+8192].to(dev).double()
M = M.float(); print("M:", tuple(M.shape))
pos_t = F.normalize(torch.from_numpy(P_sfr).to(dev), dim=-1); neg_t = F.normalize(torch.from_numpy(N_sfr).to(dev), dim=-1)
FI, SCORE = {}, {}
with torch.no_grad():
    for j, f in enumerate(FIND):
        x = clear_pre(Image.open(PATH[f]).convert("RGB")).unsqueeze(0).to(dev)      # same preprocessing as the registered run
        fi = F.normalize(clear_model.encode_image(x).float(), dim=-1); FI[f] = fi
        rep = F.normalize(fi @ M, dim=-1)
        p, n = (rep @ pos_t[j]).item(), (rep @ neg_t[j]).item()
        SCORE[f] = float(np.exp(p) / (np.exp(p) + np.exp(n)))
        diff = abs(SCORE[f] - SELECTION[f]["score"])
        print(f"{f}: recomputed {SCORE[f]:.6f} | registered {SELECTION[f]['score']:.6f} | abs diff {diff:.2e} | threshold {SELECTION[f]['thr']:.6f}")
        assert diff < 2e-5, "recomputed score does not match the registered run: stop and report"
run_info["score_check"] = {f: [SCORE[f], SELECTION[f]["score"]] for f in FIND}


In [ ]:
# 4. Concept contributions (Amendment 1 formula): contribution_i = cos(image, concept_i) x SFR_i . (positive - negative prompt)
Cn = F.normalize(C_clip, dim=1).to(dev)
P1 = F.normalize(torch.from_numpy(P_sfr), dim=-1).numpy(); N1 = F.normalize(torch.from_numpy(N_sfr), dim=-1).numpy()
Wc = torch.from_numpy((C_llm @ (P1 - N1).T).astype(np.float32)).to(dev)            # concepts x findings, as in Amendment 1
EX = {}
with torch.no_grad():
    for j, f in enumerate(FIND):
        s = (FI[f] @ Cn.T).flatten()                                                  # similarity to every concept
        contrib = s * Wc[:, j]                                                        # Amendment 1: contrib = s * Wc[:, j]
        u = FI[f] @ M                                                                 # unnormalised concept-space vector
        lhs = float(contrib.double().sum() / u.norm().double()); rhs = float((F.normalize(u, dim=-1) @ (pos_t[j] - neg_t[j])).item())
        print(f"{f}: sum of contributions / norm {lhs:.6e} | score margin {rhs:.6e}")
        assert abs(lhs - rhs) <= 0.01 * abs(rhs) + 1e-6, (f, lhs, rhs)                  # contributions add up to the score margin
        top = torch.topk(contrib, 10).indices.cpu().numpy()
        rows = [{"rank": k + 1, "concept": bank.concepts[i], "contribution": float(contrib[i]),
                 "relevant": bool(re.search(RELEVANCE[f], bank.concepts[i], re.I))} for k, i in enumerate(top)]
        EX[f] = {"image": SELECTION[f]["image"], "labels": entry.loc[SELECTION[f]["image"], "Finding Labels"],
                 "score": SCORE[f], "threshold": SELECTION[f]["thr"], "margin_check": [lhs, rhs],
                 "contribution_sum": float(contrib.sum()), "relevant_share_top10": float(np.mean([r["relevant"] for r in rows])),
                 "top10": rows}
        Image.open(PATH[f]).convert("L").resize((512, 512), Image.LANCZOS).save(f"{OUT}/{f}_{SELECTION[f]['image']}")


In [ ]:
# 5. Optional: does any VinDr-CXR DICOM header record ethnicity? (runs only if the VinBigData competition is attached)
from concurrent.futures import ThreadPoolExecutor
VIN = None
for root, dirs, files in os.walk("/kaggle/input"):
    if "train.csv" in files and "train" in dirs and os.path.exists(os.path.join(root, "train")):
        VIN = root; break
    dirs[:] = [d for d in dirs if d not in ("train", "test") and not d.startswith("images")]
if VIN is None:
    run_info["vindr_ethnicity"] = "not checked (VinBigData not attached)"; print(run_info["vindr_ethnicity"])
else:
    import pydicom
    paths = sorted(glob.glob(os.path.join(VIN, "train", "*.dicom")))
    def head(p):
        d = pydicom.dcmread(p, stop_before_pixels=True)
        eg = d.get((0x0010, 0x2160)); val = str(eg.value).strip() if eg is not None else ""
        return val, {e.keyword for e in d if e.keyword}
    with ThreadPoolExecutor(16) as ex: H = list(ex.map(head, paths))
    vals = pd.Series([h[0] for h in H]); kw = set().union(*[h[1] for h in H])
    run_info["vindr_ethnicity"] = {"n_headers": len(H), "ethnic_group_nonempty": int((vals != "").sum()),
                                   "ethnic_group_values": vals[vals != ""].value_counts().head(10).to_dict(),
                                   "keywords_like_race_or_ethnic": sorted(k for k in kw if re.search("ethnic|race|racial", k, re.I)),
                                   "all_patient_keywords": sorted(k for k in kw if k.startswith("Patient"))}
    print(json.dumps(run_info["vindr_ethnicity"], indent=1))


In [ ]:
# 6. Save and print the summary
run_info["total_min"] = round((time.time() - T0) / 60, 1)
json.dump({"run_info": run_info, "examples": EX}, open(f"{OUT}/fig7_results.json", "w"), indent=1)
!cd /kaggle/working && rm -f fig7_outputs.zip && zip -qr fig7_outputs.zip fig7_out
print("===== FIG7 SUMMARY =====")
for f in FIND:
    e = EX[f]
    print(f"{f} | {e['image']} | labels {e['labels']} | score {e['score']:.6f} (threshold {e['threshold']:.6f}) | relevant in top 10: {e['relevant_share_top10']:.1f}")
    for r in e["top10"]:
        print(f"   {r['rank']:>2}. {'R' if r['relevant'] else '-'} {r['contribution']:+.5f}  {r['concept']}")
print("VinDr ethnicity check:", run_info["vindr_ethnicity"] if isinstance(run_info["vindr_ethnicity"], str) else
      {k: run_info["vindr_ethnicity"][k] for k in ["n_headers", "ethnic_group_nonempty", "keywords_like_race_or_ethnic"]})
print("minutes:", run_info["total_min"])
print("===== END =====")
